# Notebook 02 — EDA: Campaign Performance Analysis
**Goal:** Understand which channels, months, job types, and contact frequencies drive conversion.
Translate raw data patterns into business-facing insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import sys
sys.path.append('../src')
from eda_utils import conversion_rate_bar, distribution_plot, funnel_chart, save, summary_stats

df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Loaded {len(df):,} records')
summary_stats(df)

## 1. Campaign Funnel

In [ ]:
total = len(df)
contacted = total  # all records were contacted
multi_contact = (df['campaign'] > 1).sum()
converted = df['subscribed'].sum()

stages = {
    'Total Records': total,
    'Contacted (>=1x)': contacted,
    'Contacted (>1x)': multi_contact,
    'Subscribed': converted
}
fig = funnel_chart(stages)
save(fig, '01_campaign_funnel.png')
plt.show()

## 2. Conversion Rate by Channel (Contact Type)

In [ ]:
fig = conversion_rate_bar(df, 'contact', title='Conversion Rate by Contact Channel')
save(fig, '02_conv_by_channel.png')
plt.show()

# Print exact numbers
ch = df.groupby('contact')['subscribed'].agg(['mean','count'])
ch.columns = ['conv_rate','n']
ch['conv_rate'] = (ch['conv_rate']*100).round(2)
print(ch)

## 3. Conversion Rate by Month

In [ ]:
month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
month_data = df.groupby('month')['subscribed'].mean().reindex(month_order).dropna()

fig, ax = plt.subplots(figsize=(11,5))
ax.plot(month_data.index, month_data.values*100, marker='o', color='#1F4E79', linewidth=2.5, markersize=8)
ax.fill_between(range(len(month_data)), month_data.values*100, alpha=0.15, color='#1F4E79')
ax.set_xticks(range(len(month_data)))
ax.set_xticklabels([m.title() for m in month_data.index])
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Monthly Campaign Conversion Rate', fontsize=13, fontweight='bold')
ax.set_ylabel('Conversion Rate (%)')
for i, v in enumerate(month_data.values):
    ax.text(i, v*100+0.3, f'{v*100:.1f}%', ha='center', fontsize=8)
plt.tight_layout()
save(fig, '03_conv_by_month.png')
plt.show()

## 4. Contact Frequency vs Conversion Rate

In [ ]:
freq = df[df['campaign'] <= 10].groupby('campaign')['subscribed'].agg(['mean','count']).reset_index()
freq.columns = ['contacts','conv_rate','n']
freq['conv_rate'] *= 100

fig, ax1 = plt.subplots(figsize=(10,5))
ax2 = ax1.twinx()
ax1.bar(freq['contacts'], freq['n'], color='#9DC3E6', alpha=0.6, label='Volume')
ax2.plot(freq['contacts'], freq['conv_rate'], marker='o', color='#1F4E79', linewidth=2.5, label='Conv Rate')
ax1.set_xlabel('Number of Contacts During Campaign')
ax1.set_ylabel('Number of Records', color='#9DC3E6')
ax2.set_ylabel('Conversion Rate (%)', color='#1F4E79')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.set_title('Contact Frequency: Volume vs Conversion Rate', fontsize=13, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper right')
plt.tight_layout()
save(fig, '04_contact_frequency.png')
plt.show()

print('\nKey Insight: Diminishing returns after N contacts')
print(freq[['contacts','conv_rate','n']].to_string(index=False))

## 5. Conversion by Job Type & Education

In [ ]:
fig = conversion_rate_bar(df, 'job', title='Conversion Rate by Occupation')
save(fig, '05_conv_by_job.png')
plt.show()

In [ ]:
fig = conversion_rate_bar(df, 'education', title='Conversion Rate by Education Level')
save(fig, '06_conv_by_education.png')
plt.show()

## 6. Age Distribution by Outcome

In [ ]:
fig = distribution_plot(df, 'age')
save(fig, '07_age_distribution.png')
plt.show()

fig = conversion_rate_bar(df, 'age_group', title='Conversion Rate by Age Group')
save(fig, '08_conv_by_age_group.png')
plt.show()

## 7. Economic Context vs Conversion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, col, label in zip(axes,
    ['euribor3m', 'cons_conf_idx'],
    ['Euribor 3M Rate', 'Consumer Confidence Index']):
    for val, lbl, color in zip([0,1], ['Not Subscribed','Subscribed'], ['#9DC3E6','#1F4E79']):
        df[df['subscribed']==val][col].plot.kde(ax=ax, label=lbl, color=color, linewidth=2)
    ax.set_title(f'{label} by Outcome', fontweight='bold')
    ax.legend()
plt.tight_layout()
save(fig, '09_economic_context.png')
plt.show()

## 8. Key EDA Findings Summary

In [ ]:
findings = {
    'Overall Conversion Rate': f"{df['subscribed'].mean()*100:.2f}%",
    'Best Channel': df.groupby('contact')['subscribed'].mean().idxmax(),
    'Best Month': df.groupby('month')['subscribed'].mean().idxmax(),
    'Best Job Segment': df.groupby('job')['subscribed'].mean().idxmax(),
    'Optimal Contact Count': df.groupby('campaign')['subscribed'].mean().idxmax(),
    'Previously Contacted Lift': f"{(df[df['was_contacted_before']==1]['subscribed'].mean() - df[df['was_contacted_before']==0]['subscribed'].mean())*100:.2f}pp"
}
for k, v in findings.items():
    print(f'  {k:<35}: {v}')